# EmotionAI — Google Colab Training Notebook

This notebook trains the EmotionAI CNN on the FER2013 dataset using Colab's GPU.

## Prerequisites
1. Upload the FER2013 folder structure to Google Drive:
   ```
   MyDrive/EmotionAI/train/<class>/*.jpg
   MyDrive/EmotionAI/test/<class>/*.jpg
   MyDrive/EmotionAI/shared/labels.json
   ```
2. Runtime → Change runtime type → **GPU (T4)**
3. Run all cells top to bottom.
4. Download `emotion_cnn.keras` and `metadata.json` at the end.

> **Scientific note:** This system classifies visible facial expressions — not internal emotional state.

In [ ]:
# ── Cell 1: Mount Google Drive ──────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os
BASE_DIR = '/content/drive/MyDrive/EmotionAI'
TRAIN_DIR = f'{BASE_DIR}/train'
TEST_DIR  = f'{BASE_DIR}/test'
SHARED_LABELS = f'{BASE_DIR}/shared/labels.json'

assert os.path.isdir(TRAIN_DIR), f'train/ not found at {TRAIN_DIR}'
assert os.path.isdir(TEST_DIR),  f'test/ not found at {TEST_DIR}'
assert os.path.isfile(SHARED_LABELS), f'labels.json not found at {SHARED_LABELS}'
print('Drive mounted and paths verified ✓')

In [ ]:
# ── Cell 2: Install/check dependencies ─────────────────────────────────────
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'scikit-learn', 'Pillow', 'numpy', 'pandas', 'matplotlib', 'tqdm'],
               check=True)

import tensorflow as tf
print(f'TensorFlow: {tf.__version__}')
print(f'GPU: {tf.config.list_physical_devices("GPU") or "None — switch runtime to GPU"}')

In [ ]:
# ── Cell 3: Dataset inspection ──────────────────────────────────────────────
import json
from pathlib import Path

with open(SHARED_LABELS) as f:
    labels = json.load(f)

print('Class mapping:', labels['class_to_index'])
print('Image spec:', labels['image'])
print()

for split_name, split_dir in [('train', TRAIN_DIR), ('test', TEST_DIR)]:
    print(f'── {split_name}/')
    for cls_dir in sorted(Path(split_dir).iterdir()):
        n = len(list(cls_dir.glob('*.*')))
        tag = '(excluded)' if cls_dir.name.lower() in labels.get('excluded_source_classes', []) else ''
        print(f'   {cls_dir.name:12s}: {n:5d} images {tag}')

In [ ]:
# ── Cell 4: Preprocessing — build manifests and load arrays ────────────────
import numpy as np
import pandas as pd
from PIL import Image
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm

IMG_H = labels['image']['height']  # 48
IMG_W = labels['image']['width']   # 48
SOURCE_MAP = {k.lower(): v for k, v in labels['source_folder_names'].items()}
EXCLUDED   = {c.lower() for c in labels.get('excluded_source_classes', [])}
C2I        = labels['class_to_index']

def collect_rows(root, split_name):
    rows = []
    for cls_dir in sorted(Path(root).iterdir()):
        key = cls_dir.name.lower()
        if key in EXCLUDED or key not in SOURCE_MAP:
            continue
        label = SOURCE_MAP[key]
        idx   = int(C2I[label])
        for p in cls_dir.rglob('*'):
            if p.is_file() and p.suffix.lower() in {'.jpg','.jpeg','.png'}:
                rows.append({'path': str(p), 'label': label, 'class_index': idx, 'split': split_name})
    return rows

train_rows = collect_rows(TRAIN_DIR, 'train_source')
test_rows  = collect_rows(TEST_DIR,  'test')
print(f'Train source: {len(train_rows)} | Test: {len(test_rows)}')

train_df = pd.DataFrame(train_rows)
test_df  = pd.DataFrame(test_rows)
train_part, val_part = train_test_split(
    train_df, test_size=0.15, random_state=42, stratify=train_df['class_index']
)
train_part = train_part.copy(); train_part['split'] = 'train'
val_part   = val_part.copy();   val_part['split']   = 'val'
test_df    = test_df.copy();    test_df['split']    = 'test'
manifest   = pd.concat([train_part, val_part, test_df], ignore_index=True)
print('Split counts:', manifest['split'].value_counts().to_dict())

def load_img(path):
    with Image.open(path) as im:
        im = im.convert('L').resize((IMG_W, IMG_H), Image.Resampling.BILINEAR)
        arr = np.asarray(im, dtype=np.float32) / 255.0
    return arr.reshape(IMG_H, IMG_W, 1)

def build_arrays(df, split):
    subset = df[df['split'] == split]
    X, y = [], []
    for _, row in tqdm(subset.iterrows(), total=len(subset), desc=split):
        try:
            X.append(load_img(row['path']))
            y.append(row['class_index'])
        except Exception as e:
            print(f'Skip {row["path"]}: {e}')
    return np.stack(X), np.array(y, dtype=np.int64)

X_train, y_train = build_arrays(manifest, 'train')
X_val,   y_val   = build_arrays(manifest, 'val')
X_test,  y_test  = build_arrays(manifest, 'test')
print(f'X_train: {X_train.shape}  X_val: {X_val.shape}  X_test: {X_test.shape}')

In [ ]:
# ── Cell 5: Build CNN model ─────────────────────────────────────────────────
from tensorflow import keras
from tensorflow.keras import layers

NUM_CLASSES = labels['num_classes']  # 5

def build_emotion_cnn(num_classes=5, input_shape=(48, 48, 1),
                      dropout_rate=0.4, sdrop=0.2):
    inp = keras.Input(shape=input_shape)
    x = inp
    for filters in [32, 64, 128, 256]:
        x = layers.Conv2D(filters, 3, padding='same', use_bias=False)(x)
        x = layers.BatchNormalization()(x)
        x = layers.ReLU()(x)
        x = layers.Conv2D(filters, 3, padding='same', use_bias=False)(x)
        x = layers.BatchNormalization()(x)
        x = layers.ReLU()(x)
        x = layers.MaxPooling2D()(x)
        x = layers.SpatialDropout2D(sdrop)(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(512, use_bias=False)(x); x = layers.BatchNormalization()(x); x = layers.ReLU()(x)
    x = layers.Dropout(dropout_rate)(x)
    x = layers.Dense(256, use_bias=False)(x); x = layers.BatchNormalization()(x); x = layers.ReLU()(x)
    x = layers.Dropout(dropout_rate/2)(x)
    out = layers.Dense(num_classes, activation='softmax')(x)
    return keras.Model(inputs=inp, outputs=out, name='emotion_cnn')

model = build_emotion_cnn(NUM_CLASSES)
model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)
model.summary()

In [ ]:
# ── Cell 6: Class weights + Augmentation + Training ─────────────────────────
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import time

class_weights = dict(enumerate(compute_class_weight('balanced',
    classes=np.arange(NUM_CLASSES), y=y_train)))
print('Class weights:', {labels['index_to_class'][str(k)]: round(v,3) for k,v in class_weights.items()})

datagen = ImageDataGenerator(
    horizontal_flip=True, rotation_range=10,
    zoom_range=0.10, width_shift_range=0.05, height_shift_range=0.05
)
datagen.fit(X_train)

BATCH  = 64
EPOCHS = 60

callbacks = [
    keras.callbacks.ModelCheckpoint(
        '/content/emotion_cnn.keras', monitor='val_accuracy',
        save_best_only=True, verbose=1),
    keras.callbacks.EarlyStopping(
        monitor='val_accuracy', patience=10,
        restore_best_weights=True, verbose=1),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6, verbose=1),
]

t0 = time.time()
history = model.fit(
    datagen.flow(X_train, y_train, batch_size=BATCH, seed=42),
    steps_per_epoch=len(X_train) // BATCH,
    epochs=EPOCHS,
    validation_data=(X_val, y_val),
    class_weight=class_weights,
    callbacks=callbacks,
    verbose=1,
)
print(f'Training done in {(time.time()-t0)/60:.1f} min')

In [ ]:
# ── Cell 7: Evaluation plots ────────────────────────────────────────────────
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report

best_model = keras.models.load_model('/content/emotion_cnn.keras')
test_loss, test_acc = best_model.evaluate(X_test, y_test, verbose=0)
print(f'\nTest accuracy: {test_acc:.4f} | Test loss: {test_loss:.4f}')

y_pred = np.argmax(best_model.predict(X_test, verbose=0), axis=1)
class_names = [labels['index_to_class'][str(i)] for i in range(NUM_CLASSES)]

print('\nClassification Report:')
print(classification_report(y_test, y_pred, target_names=class_names, digits=4))

# Training curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
ep = range(1, len(history.history['loss'])+1)
ax1.plot(ep, history.history['loss'], 'b-o', ms=4, label='Train')
ax1.plot(ep, history.history['val_loss'], 'r-o', ms=4, label='Val')
ax1.set_title('Loss'); ax1.legend(); ax1.grid(alpha=0.3)
ax2.plot(ep, history.history['accuracy'], 'b-o', ms=4, label='Train')
ax2.plot(ep, history.history['val_accuracy'], 'r-o', ms=4, label='Val')
ax2.set_title('Accuracy'); ax2.legend(); ax2.grid(alpha=0.3)
plt.suptitle('EmotionAI CNN Training Curves'); plt.tight_layout()
plt.savefig('/content/training_curves.png', dpi=150); plt.show()

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
cm_n = cm / cm.sum(axis=1, keepdims=True)
fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(cm_n, cmap='Blues', vmin=0, vmax=1)
fig.colorbar(im)
ax.set_xticks(range(len(class_names))); ax.set_xticklabels(class_names, rotation=30, ha='right')
ax.set_yticks(range(len(class_names))); ax.set_yticklabels(class_names)
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
ax.set_title('Confusion Matrix (Normalized)')
for i in range(len(class_names)):
    for j in range(len(class_names)):
        ax.text(j, i, f'{cm_n[i,j]:.2f}\n({cm[i,j]})', ha='center', va='center',
                color='white' if cm_n[i,j]>0.5 else 'black', fontsize=9)
plt.tight_layout()
plt.savefig('/content/confusion_matrix.png', dpi=150); plt.show()

In [ ]:
# ── Cell 8: Save metadata.json ─────────────────────────────────────────────
import json
from datetime import datetime, timezone

best_epoch = int(np.argmax(history.history['val_accuracy']))
metadata = {
    'model_name': 'emotion_cnn',
    'model_version': '0.1.0',
    'model_file': 'emotion_cnn.keras',
    'framework': 'tensorflow.keras',
    'task': 'facial_expression_recognition',
    'disclaimer': 'Classifies visible facial expressions. Does not determine internal emotional state.',
    'input': labels['image'],
    'num_classes': NUM_CLASSES,
    'class_to_index': labels['class_to_index'],
    'index_to_class': labels['index_to_class'],
    'training': {
        'dataset': 'FER2013',
        'excluded_classes': list(labels.get('excluded_source_classes', [])),
        'trained_at_utc': datetime.now(timezone.utc).isoformat(),
        'epochs_trained': len(history.history['loss']),
        'best_epoch': best_epoch + 1,
        'batch_size': BATCH,
        'augmentation': 'h-flip, rotation±10°, zoom±10%, shift±5%',
        'class_weight_strategy': 'balanced',
    },
    'evaluation': {
        'test_accuracy': float(test_acc),
        'test_loss': float(test_loss),
        'best_val_accuracy': float(max(history.history['val_accuracy'])),
        'note': 'Measured on held-out test set. Do not fabricate values.',
    }
}

with open('/content/metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print('Saved metadata.json')
print(f'Test accuracy: {test_acc:.4f}')
print(f'Best val accuracy: {max(history.history["val_accuracy"]):.4f}')

In [ ]:
# ── Cell 9: Download model files ────────────────────────────────────────────
from google.colab import files

print('Downloading emotion_cnn.keras ...')
files.download('/content/emotion_cnn.keras')

print('Downloading metadata.json ...')
files.download('/content/metadata.json')

print('Downloading training_curves.png ...')
files.download('/content/training_curves.png')

print('Downloading confusion_matrix.png ...')
files.download('/content/confusion_matrix.png')

print()
print('Next steps:')
print('  1. Copy emotion_cnn.keras → EmotionAI/models/emotion_cnn.keras')
print('  2. Copy metadata.json     → EmotionAI/models/metadata.json')
print('  3. Copy plots             → EmotionAI/data/reports/')
print('  4. Start the FastAPI backend and test /predict')